# Week 1 Day 2 Lab - Introducing UnderCover (free version)

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a free model, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 3 parts:

PART 1: Examine the data

PART 2: Run with a free model

PART 3: An Agent via LangChain

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of the lab. It uses <code>openrouter/free</code> throughout, so it needs no credit on OpenRouter. The full version with paid models is <a href="week1day2.ipynb">week1day2.ipynb</a>.<br/><br/>
            OpenRouter's free models allow 50 requests a day on an account without credit. With <code>TINY = True</code>, the default here, this lab makes about 30. <code>openrouter/free</code> picks a different free model for each request, so expect your results to vary more than mine.<br/><br/>
            Please execute each cell and compare your results with mine. Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [58]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field

## Connect to OpenRouter - and please read note here

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

### NOTE:

If this is a new OpenRouter account, then you might get an error at some point from OpenRouter like: "new accounts are limited to 20 requests per minute for this model"

If you get that error, then add this line immediately after calling an LLM:

`time.sleep(3)`

and you should be in good shape.

In [64]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


In [65]:
load_dotenv(override=True)
openrouter_api_key =os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: the Chat Completions API</h2>
            <span style="color:#118AB2;">We call models with the Chat Completions API. <code>messages</code> is a Python list of dicts, each with a <code>role</code> and <code>content</code>. The system prompt tells the model how to behave and the user prompt has the request. The reply is in <code>response.choices[0].message.content</code>.<br/><br/>
            Here's a quick test.
            </span>
        </td>
    </tr>
</table>

In [66]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = openrouter.chat.completions.create(model="openrouter/free", messages=messages)
print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by nvidia/nemotron-3-ultra-550b-a55b:free


Ahoy there, matey! Ye be askin’ the captain the oldest riddle in the seven seas, eh? *The meanin’ of the universe?*

**Arrr, pull up a barrel and listen close.**

There ain’t no X markin’ the spot on no chart for that one, nor no message in a bottle washed up on shore with the answer. The universe be a vast, trackless ocean—dark, deep, and teemin’ with beasts we ain’t even named yet.

But if this old salt’s got a spyglass to offer, here’s what I’ve read in the stars and the spray:

**1. The Journey *Be* the Treasure**
Ye don’t sail just to drop anchor. Ye sail for the wind in yer hair, the salt on yer lips, the roar of the cannon, and the quiet nights where the Milky Way shines brighter than the Queen’s crown jewels. The meanin’ ain’t the destination—it’s the *sailin’*.

**2. Freedom’s the Only Compass**
A pirate answers to no King, no Governor, no gravity even, if we can help it. The universe don’t owe ye a code, matey. *Ye write the code.* Ye chart yer own course. That freedom? That’s the only true north.

**3. Share Yer Rum (And Yer Stories)**
A ship don’t run on a solo captain. It runs on a crew—scallywags, quartermasters, powder monkeys, and the cook who makes grog taste like nectar. The meanin’ be found in the laughter in the mess hall, the backs ye watch, and the legends ye leave behind for the next generation of buccaneers.

**4. Curiosity Be Yer Cutlass**
Every island unexplored, every nebula uncharted, every question unasked—that’s booty waitin’ to be plundered. Wonder is the sharpest blade in yer belt. Never stop askin’ "What lies beyond the horizon?"

**5. Accept the Storms**
The sea *will* try to swallow ye whole. Krakens rise. Masts snap. Friends walk the plank. The universe is indifferent, cold, and cruel. But a true pirate? We laugh in the face of the tempest, patch the hull, and keep sailin’. *Defiance* is its own kind of meanin’.

***

So hoist the colors, matey! The meanin’ of the universe ain't somethin' ye *find* buried in the sand.

**It’s the flag ye choose to fly while ye dance upon the deck of a spinnin’ rock in the middle of nowhere.**

Now, pass the rum! I’ve got a heading to set for the Andromeda Galaxy—heard the space-whales there sing a mighty fine shanty. 🏴‍☠️🌌🥃

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

The queue has 25 claims. This version sets `TINY = True` to keep 5 of them, which fits the free daily limit. Set it to `False` for all 25 if your account allows more requests.

In [9]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [10]:
TINY = True

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

5 claims, 5 labels, 50 policies
The policy wording is 1,369 words


In [11]:
claims[0]

{'claim_id': 'CLM-1007',
 'policy_id': 'POL-2007',
 'customer_id': 'CUS-3007',
 'claim_type': 'baggage',
 'submitted_on': '2026-07-22',
 'incident_date': '2026-07-13',
 'incident_location': 'Stockholm, Sweden',
 'summary': 'Phone dropped in the hotel shower',
 'story': 'I dropped my phone in the shower at my hotel in Stockholm on July 13. I know how that sounds. It was on the soap shelf playing music, I was singing along, and during the key change I knocked it off the shelf. It landed screen down in about an inch of water and stayed there longer than it should have, because I had shampoo in my eyes. A repair shop looked at it the next day and said the liquid damage was too extensive to repair. I am claiming the cost of the replacement phone, which was $1,099.',
 'items': [{'description': 'Smartphone, 256 GB, replacement cost',
   'category': 'electronics',
   'amount': 1099.0}],
 'amount_claimed': 1099.0,
 'documents': [{'name': 'Repair shop report',
   'type': 'repair_report',
   'con

Here is the handler's decision on that claim which we'll use to validate the LLM results.

In [12]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1007',
 'decision': 'approve',
 'payout': 50.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-07-24',
 'reason': 'Accidental damage during the covered trip, with a repair report stating the phone cannot be repaired. The policy is Economy with no Electronics Plus, so the phone counts at the $150 per-item limit, and the $100 deductible comes off.',
 'reference': 'Policy wording, sections 5 and 10',
 'calculation': [{'step': 'Phone at the per-item limit', 'amount': 150.0},
  {'step': 'Less the deductible', 'amount': -100.0}],
 'tests': 'Payout arithmetic: a per-item limit far below the amount claimed',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'amusing',
 'tiny': True}

### Explore the data

It's always good to dig into the data before running experiments.

In [13]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
1,CLM-1015,Flew home from Paris on arrival after leaving ...,trip_interruption,2340.0,routine,decline,0.0
2,CLM-1017,Backpack snatched by a scooter rider in Naples,baggage,830.0,routine,refer,0.0
3,CLM-1020,Prescription sunglasses stolen by a monkey at ...,baggage,240.0,manual,approve,240.0
4,CLM-1021,Tote bag snatched from her shoulder in Buenos ...,baggage,1103.0,hard,approve,825.0


Claim CLM-1015 might be a familiar example...

In [14]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


Here's another one that might ring a bell...

In [30]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1007"][0]
decision = queue_labels["CLM-1007"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Phone dropped in the hotel shower**

I dropped my phone in the shower at my hotel in Stockholm on July 13. I know how that sounds. It was on the soap shelf playing music, I was singing along, and during the key change I knocked it off the shelf. It landed screen down in about an inch of water and stayed there longer than it should have, because I had shampoo in my eyes. A repair shop looked at it the next day and said the liquid damage was too extensive to repair. I am claiming the cost of the replacement phone, which was $1,099.

**Handler:** approve $50.00. Accidental damage during the covered trip, with a repair report stating the phone cannot be repaired. The policy is Economy with no Electronics Plus, so the phone counts at the $150 per-item limit, and the $100 deductible comes off.


This chart counts the claims of each type, split by the handler's decision.

In [16]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [17]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with a free LLM

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: Structured Outputs</h2>
            <span style="color:#118AB2;">With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to <code>parse()</code> as the <code>response_format</code>. The reply comes back as an object of that class in <code>response.choices[0].message.parsed</code>.
            </span>
        </td>
    </tr>
</table>

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

In [19]:
class ClaimDecision(BaseModel):

    rationale: str =Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision : Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

In [27]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""
def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]


### Calling the LLM

In [ ]:
def decide(claim, model):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "seconds": round(time.time() - start, 1),
    }

In [28]:
def decide(claim,model):
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed

    return{
        "model": model,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "seconds": round(time.time() - start, 1),
    }


In [29]:
first = decide(claims[0], "openrouter/free")
first

{'model': 'openrouter/free',
 'technique': 'LLM',
 'claim_id': 'CLM-1007',
 'decision': 'approve',
 'payout': 50.0,
 'rationale': 'The claim is for accidental damage to a smartphone during the covered trip in Stockholm. Section 5 covers accidental damage to personal belongings during the trip, and the repair shop report states the phone is beyond economical repair, which satisfies the document requirement for accidental damage. The claim was filed within 90 days of the incident.\n\nHowever, the customer is on the Economy plan and did not purchase the Electronics Plus add-on. The Economy per-item limit is $150, so the eligible amount for the phone is capped at $150, not the claimed $1,099. The $100 deductible applies once to Baggage and Personal Effects.\n\nCalculation under Section 10: eligible amount $150 − $0 other payments − $100 deductible = $50. This is below the Economy Baggage and Personal Effects maximum of $750.',
 'seconds': 19.8}

### Now running all the claims

In [31]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model}: {right} of {len(results)} right in {seconds:.0f} seconds")
    return results

In [33]:
free_results = run("openrouter/free", claims, labels)

openrouter/free: 4 of 5 right in 71 seconds


## Results by difficulty

This counts the claims the model got right at each level of difficulty. Expect the manual claims to be wrong for now.

In [34]:
scored = pd.DataFrame(free_results)
scored.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum")

difficulty,hard,manual,routine
model,,,
openrouter/free,1,0,3


## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Pass in another set of results to see its misses.

In [35]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(free_results, labels)

**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The claim is for loss/theft of prescription sunglasses under the Baggage and Personal Effects benefit. The incident occurred during the covered trip, the amount ($240) is within the First plan's $500 per-item limit, and the $0 deductible applies. However, section 5 requires that theft be reported to the local police within 24 hours and that a copy of the police report be provided. The customer did not obtain a police report (the police post declined to take a monkey report) and instead supplied a guide letter and optician receipt. Because a required document is missing, and the wording does not settle whether a guide letter can substitute for a police report or whether a monkey taking property qualifies as theft under the policy, the claim should be referred to a colleague rather than approved or declined outright.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Accuracy, cost and speed are the numbers a customer will ask about. Cost per claim is the easiest one to relate to the business: compare it with what a handler's time on a claim costs.<br/><br/>
            The kind of mistake matters too. A wrong approval pays out money the policy doesn't owe, a wrong decline lets a customer down and a wrong referral costs a handler's time.
            </span>
        </td>
    </tr>
</table>

## Part 3: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [36]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

Arr, matey, that be the grandest question on the Seven Cosmic Seas—and there be no single treasure map answer.

From the telescope’s view: the universe don’t come with a sign sayin’ “Here be the meaning.” It just be—vast, ancient, and followin’ its own laws.

From the philosopher’s view: meanin’ may not be hidden in the stars at all; it be somethin’ we creatures forge—through love, curiosity, courage, kindness, and the tales we tell.

From the pirate’s view: if the cosmos won’t hand ye a purpose, then hoist yer own flag. Sail toward what gives yer life weight, leave a bit more good in yer wake, and don’t be so busy askin’ the sea fer meanin’ that ye forget to sail it.

So: the universe’s meaning may be a riddle with no fixed answer—but yer part in it be written by what ye choose to do. Aye, that’s the wind in the sails.

In [71]:
recorded = {}


class BaggageItem(BaseModel):
    name: str = Field(description="Short name of the item")
    claimed: float = Field(description="Amount the customer claims for this item, in USD")
    limit: float = Field(description="Per-item limit that applies to this item, in USD")


@tool
def calculate_baggage_payout(
    items: list[BaggageItem],
    deductible: float,
    paid_by_others: float = 0.0,
) -> str:
    """Compute the baggage payout from the items. For each item you decide the applicable
    per-item limit from the wording; this tool only does the arithmetic."""
    eligible = sum(min(i.claimed, i.limit) for i in items)
    payout = max(eligible - paid_by_others - deductible, 0)
    return json.dumps({"eligible": eligible, "payout": payout})


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses `openrouter/free` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with the single calls in Part 2.

In [80]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

claims_manual = (DATA / "claims_manual.md").read_text(encoding="utf-8")



AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

Work item by item. For each item, write down: the item, the amount claimed,
and which per-item limit applies. An add-on changes a limit only for the items
its definition in the wording lists. The category on the claim form (for example
"electronics") is the customer's label, not the policy's. Check every item
against the add-on's own definition before applying a higher limit.

{policy_wording}

{claims_manual}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=llm, tools=[get_policy, record_decision,calculate_baggage_payout], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [74]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

KeyboardInterrupt: 

Now all the claims. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. A claim with no recorded decision counts as wrong.

In [81]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    return {
        "model": "openrouter/free",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day: https://openrouter.ai/settings/credits?utm_source=api&utm_campaign=free_daily_limit', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1795392000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day: https://openrouter.ai/settings/credits?utm_source=api&utm_campaign=free_daily_limit'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day: https://openrouter.ai/settings/credits?utm_source=api&utm_ca

In [76]:
pd.DataFrame(free_results + agent_results).groupby(["model", "technique"], sort=False)[["right", "seconds"]].sum()

right  seconds
model           technique                
openrouter/free LLM            4     70.9
                Agent          4    214.5

In [77]:
show_misses(agent_results, labels)

**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The theft of personal items is covered under Baggage and Personal Effects, with a police report filed within 24 hours and receipts provided for the electronics. The Electronics Plus add‑on raises the per‑item limit for electronics to $1,200, so the claimed amounts are fully eligible. After applying the $50 deductible, the payable amount is $1,053, which is below the $1,500 maximum.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


In [79]:
raw = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key).chat.completions.with_raw_response.create(
    model="openrouter/free",
    messages=[{"role": "user", "content": "hi"}],
    max_tokens=5,
)
print({k: v for k, v in raw.headers.items() if "ratelimit" in k.lower()})

{}


In [78]:
import requests

r = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers={"Authorization": f"Bearer {openrouter_api_key}"},
)
print(json.dumps(r.json(), indent=2))

{
  "data": {
    "label": "sk-or-v1-ee6...3ed",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 100,
    "limit_reset": null,
    "limit_remaining": 100,
    "include_byok_in_limit": false,
    "budgets": [
      {
        "scope": "key_limit",
        "guardrail_id": null,
        "budget_id": null,
        "limit_usd": 100,
        "spend_usd": 0,
        "remaining_usd": 100,
        "reset_interval": null,
        "resets_at": null,
        "include_byok": false
      }
    ],
    "effective_budget": {
      "scope": "key_limit",
      "guardrail_id": null,
      "budget_id": null,
      "limit_usd": 100,
      "spend_usd": 0,
      "remaining_usd": 100,
      "reset_interval": null,
      "resets_at": null,
      "include_byok": false
    },
    "usage": 0,
    "usage_daily": 0,
    "usage_weekly": 0,
    "usage_monthly": 0,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": t

## Putting it all together

Here is every experiment in one table. Speed is the average time for one claim.

In [43]:
rows = []
for results in [free_results, agent_results]:
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,technique,accuracy,speed
0,openrouter/free,LLM,4 / 5,14.2 seconds a claim
1,openrouter/free,Agent,3 / 5,27.2 seconds a claim


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand their accuracy and speed.<br/><br/>
            <code>openrouter/free</code> picks a different free model for each request. The free models on OpenRouter each have an id that ends in <code>:free</code>.<br/><br/>
            Challenge: use a specific free model in place of <code>openrouter/free</code>. Which free model gets the most claims right with the agent?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations! And if I could disturb you with a quick request:</h2>
            <span style="color:#069D75;">You've finished the first main lab. You tried a free model and an agent on UnderCover's claims, with accuracy and speed side by side.<br/><br/>
            On Day 3 we bring in coding agents and start building the UnderCover platform.<br/><br/>
            <b>I have a quick request for you. If you have a moment to rate the course, I'd hugely appreciate it. It's one of the main ways Udemy decides whether
            to show the course to others, and I'd be super grateful.</b>
            </span>
        </td>
    </tr>
</table>